In [52]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB, ComplementNB
from sklearn.metrics import classification_report, confusion_matrix

RANDOM_SEED = 0

In [53]:
dataset = pd.read_csv("spam.csv", encoding="ISO-8859-1")
dataset.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [54]:
data = dataset[["v1", "v2"]]
data.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [55]:
data = data.rename(columns={"v1": "SPAM", "v2": "MESSAGE"})
classes_encoding = {"spam": 1, "ham": 0}
data["SPAM"] = data["SPAM"].map(lambda x: classes_encoding[x])
data.head()

,SPAM,MESSAGE
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [56]:
X = data["MESSAGE"]
y = data["SPAM"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=RANDOM_SEED)

# Bernoulli NB

In [57]:
bow = CountVectorizer(stop_words="english", max_features=1000)
X_train_bow = bow.fit_transform(X_train.tolist())
X_test_bow = bow.transform(X_test.tolist())

bernNB = BernoulliNB()
bernNB.fit(X_train_bow, y_train)
report = classification_report(y_test, bernNB.predict(X_test_bow))
print(report)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1434
           1       0.99      0.89      0.94       238

    accuracy                           0.98      1672
   macro avg       0.99      0.94      0.96      1672
weighted avg       0.98      0.98      0.98      1672



In [58]:
confusion_matrix(y_test, bernNB.predict(X_test_bow))

array([[1432,    2],
       [  27,  211]])

# Multinomial NB

In [59]:
tfid = TfidfVectorizer(stop_words="english", max_features=1000)
X_train_tfid = tfid.fit_transform(X_train.tolist())
X_test_tfid = tfid.transform(X_test)

multiNB = MultinomialNB()
multiNB.fit(X_train_tfid, y_train)
print(classification_report(y_test, multiNB.predict(X_test_tfid)))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1434
           1       0.97      0.87      0.92       238

    accuracy                           0.98      1672
   macro avg       0.97      0.93      0.95      1672
weighted avg       0.98      0.98      0.98      1672



In [60]:
confusion_matrix(y_test, multiNB.predict(X_test_tfid))

array([[1428,    6],
       [  32,  206]])

# Complement NB

In [61]:
compNB = ComplementNB()
compNB.fit(X_train_tfid, y_train)

print(classification_report(y_test, compNB.predict(X_test_tfid)))

              precision    recall  f1-score   support

           0       1.00      0.94      0.97      1434
           1       0.74      0.98      0.84       238

    accuracy                           0.95      1672
   macro avg       0.87      0.96      0.91      1672
weighted avg       0.96      0.95      0.95      1672



In [62]:
confusion_matrix(y_test, compNB.predict(X_test_tfid))

array([[1350,   84],
       [   4,  234]])

## Proviamo il Bernoulli NB

In [77]:
messages = [
    "Hi bud, how are you?",
    "Long time no see Antonio, meet me up in Turin this summer?",
    "Congratulations! You have been selected for an Eclusive Gift Lancôme Dear Sephora Client, you have been randomly selected to receive our exclusive Lancôme Luxury Beauty Box - completely free!",
    "Hi Antonio, it' time to meet up and start together this new adventure. We will wait for you 03/11/2025 at 09.00 in our site in Turin: Turin, via Nizza 262, int.27.  When you will arrive at our desk we will ask you to follow the procedures for registration with our security team and wait for HR.",
    "Antonio If you’re thinking of travelling this year or next then today is your lucky day! Because when you go here now you can get an extra special deal on your car hire. Early bird discounts normally start in January, but this year we decided to start even earlier but you need to be quick. These discounts are available today only. And if you're worried it's too early to book for next year - don't. Even if you're not 100% sure of your travel plans you can't lose out. Because you get FREE CANCELLATIONS and FREE MODIFICATIONS on most bookings. Total flexibility for the cheapest price. I can’t make it fairer than that. Especially when you take into account that you can also: Choose a rental company located inside the airport making it quicker to collect your car Get a free additional driver on many offers Choose a full/full fuel policy to save money It really is win-win. If you’re quick, that is. As I said, you need to book today… After that you’ll pay more."
]

messages_bow = bow.transform(messages)

In [78]:
messages_pred = bernNB.predict(messages_bow)
messages_pred

array([0, 0, 1, 1, 1])